In [57]:
"""
====================================================
Download ERA5 Data
====================================================
"""

'\n====================================================\nDownload ERA5 Data\n====================================================\n'

In [58]:
#######################
#DIRECTORIES

In [59]:
#SETTING UP DIRECTORIES
mainDirectory = '/mnt/lustre/koa/koastore/torri_group/air_directory/Projects/Regional-MPAS-Project/'
print(mainDirectory)
dataDirectory=mainDirectory+"DownloadData/DATA/ERA5_Data/"

/mnt/lustre/koa/koastore/torri_group/air_directory/Projects/Regional-MPAS-Project/


In [60]:
#######################
#LIBRARIES, FUNCTIONS, and CLASSES

In [61]:
#IMPORT LIBRARIES
# --- Add your Functions folder to sys.path ---
import sys
path = mainDirectory + '/Libraries/'
sys.path.append(path)


# --- Import all your function modules ---
import importlib
modules = [
    "Libraries",
]

for mod in modules:
    globals()[mod] = importlib.import_module(mod)        # import module itself
    globals().update(vars(globals()[mod]))              # import all functions into global namespace

In [62]:
#IMPORT FUNCTIONS
# --- Add your Functions folder to sys.path ---
import sys
path = mainDirectory + 'Functions_2.0/'
sys.path.append(path)


# --- Import all your function modules ---
import importlib
modules = [
    "AreaAverageFunctions",
    "ComputationFunctions",
    "DataFunctions",
    "DerivativeFunctions",
    "PlottingFunctions",
    "StatisticalFunctions",
]
for mod in modules:
    globals()[mod] = importlib.import_module(mod)        # import module itself
    globals().update(vars(globals()[mod]))              # import all functions into global namespace

In [63]:
#IMPORT CLASSES
# --- Add your Functions folder to sys.path ---
import sys
path = mainDirectory + 'Functions_2.0/Classes/'
sys.path.append(path)


# --- Import all your function modules ---
import importlib
modules = [
    "Classes_1",
]
for mod in modules:
    globals()[mod] = importlib.import_module(mod)        # import module itself
    globals().update(vars(globals()[mod]))              # import all functions into global namespace

In [40]:
###########################
#DOWNLOADING DATA FUNCTIONS

In [41]:
#DOWNLOADING ERA5

#"Download ERA data with python" Code Inspired by https://github.com/joaohenry23/Download_ERA5_with_python
# https://cds.climate.copernicus.eu/how-to-api

# pip install cdsapi
# pip install "cdsapi>=0.7.4"
import cdsapi
from tqdm import tqdm
import os

def DownloadERA5(variables,years,months,days,area):
    c = cdsapi.Client()    
    for variable in tqdm(variables, desc="Downloading ERA5 variables"):
        print(f"Downloading {variable}","\n")
        c.retrieve(
            'reanalysis-era5-pressure-levels',
            {
                'product_type': 'reanalysis',
                'format': 'netcdf',
                'variable': variable,
                'pressure_level': [
                    '100', '250', '500', '750', '1000'
                ],
                'year': years,
                'month': months,
                'day': days,
                'time': [f"{h:02d}:00" for h in range(24)],
                'area': area,
                'grid': [0.25, 0.25],
            },
            os.path.join(dataDirectory,date_folder,f'{variable}_ERA5_{date_folder}.nc') 
        )

In [49]:
#DATE INFORMATION
def MakeDateFolder(date_string):
    date_folder = strings.DateString(date_string)
    #adding date to output folder
    subdir = os.path.join(dataDirectory, date_folder)
    os.makedirs(subdir, exist_ok=True)
    return date_folder

#COORDINATES INFORMATION
def GetCoordinates(longitude,latitude, dx_m=250e3, dy_m=250e3,grid_res=0.25):
    longitude = coordinates.DMSToDecimal(*longitude)
    latitude = coordinates.DMSToDecimal(*latitude)
    
    dlon=coordinates.dxTOdlon(dx_m=dx_m, lat_deg=latitude)
    dlat=coordinates.dyTOdlat(dy_m=dy_m)
    
    N, W, S, E = [latitude + dlat, longitude - dlon,
                  latitude - dlat, longitude + dlon]
    print("Coords box:", [N, W, S, E])
    # Round outward to 0.25 grid
    N = math.ceil(N / grid_res) * grid_res     # round north up
    S = math.floor(S / grid_res) * grid_res    # round south down
    W = math.floor(W / grid_res) * grid_res    # round west down (more negative)
    E = math.ceil(E / grid_res) * grid_res     # round east up

    area=[N,W,S,E]
    print("Rounded box:", area)
    return area

#VARIABLES INFORMATION
def GetVariableNames():
    variables = [
        'u_component_of_wind','v_component_of_wind','vertical_velocity',
        'divergence',
        'vorticity',
        'temperature',
        'specific_humidity','specific_cloud_liquid_water_content',
        'specific_cloud_ice_water_content','specific_rain_water_content',
        'relative_humidity','cloud_cover','geopotential'
    ]
    return variables

In [43]:
###########################
#DOWNLOADING TRACER DATA

In [44]:
#coorindates information
#GETTING BOUNDING BOX centered at Houston, TX Mobile Facility (TRACER) Facility S2 ==> CSAP (C-Band Scanning ARM Precipitation Radar)
# 29°31'55"N, 95°17'2"W
longitude = (95,17,2,'W')
latitude = (29,31,55,'N')
area=GetCoordinates(longitude,latitude)
variables=GetVariableNames()

Coords box: [31.78024845924127, -97.86790574593715, 27.28364042964762, -92.69987203184061]
Rounded box: [32.0, -98.0, 27.25, -92.5]


In [45]:
###########################
#DATE ONE (BORING CASE)

In [52]:
#INFORMATION
#date information
date_string = "06-08 - 06-10 (2022)"
years=['2022']
months=['06']
days=['08','09','10'] #middle date is simulation date
date_folder=MakeDateFolder(date_string)

#running
DownloadERA5(variables,years,months,days,area)

2025-09-03 18:27:11,292 INFO [2024-09-26T00:00:00] Watch our [Forum](https://forum.ecmwf.int/) for Announcements, news and other discussed topics.

2025-09-03 18:27:11,820 INFO Request ID is e47243cb-1ab3-426b-9895-dfceccbf1216
2025-09-03 18:27:12,049 INFO status has been updated to accepted
2025-09-03 18:27:21,066 INFO status has been updated to running
2025-09-03 18:27:26,373 INFO status has been updated to successful


f30f85a94d1e9e07dcbeef27926800c3.nc:   0%|          | 0.00/382k [00:00<?, ?B/s]

2025-09-03 18:27:29,751 INFO Request ID is d8a5582e-99ae-44d0-9a25-dd7beb483a10
2025-09-03 18:27:29,987 INFO status has been updated to accepted
2025-09-03 18:27:44,346 INFO status has been updated to running
2025-09-03 18:28:47,051 INFO status has been updated to successful


bee687136d770464e5a533256bb632a3.nc:   0%|          | 0.00/387k [00:00<?, ?B/s]

...

d10b66eaa904368b07b23eccb1b2f110.nc:   0%|          | 0.00/375k [00:00<?, ?B/s]

2025-09-03 18:37:38,677 INFO Request ID is c24c6e07-1a4e-4dd7-a26a-318b3fc0f850
2025-09-03 18:37:38,920 INFO status has been updated to accepted
2025-09-03 18:37:48,099 INFO status has been updated to running
2025-09-03 18:38:30,216 INFO status has been updated to successful


23dd90cca0e1e55f901942049f10a5af.nc:   0%|          | 0.00/62.2k [00:00<?, ?B/s]

2025-09-03 18:38:33,583 INFO Request ID is 41fa0a30-8046-43e0-92de-bdaae6fe718c
2025-09-03 18:38:33,849 INFO status has been updated to accepted
2025-09-03 18:38:48,247 INFO status has been updated to running
2025-09-03 18:39:25,134 INFO status has been updated to successful


375434b1808631b77bb7c0d4a121b8d.nc:   0%|          | 0.00/261k [00:00<?, ?B/s]

In [53]:
###########################
#DATE TWO (RAINY CASE)

In [54]:
#INFORMATION
#date information
date_string = "06-30 - 07-02 (2022)"
years=['2022']
months=['06','07']
days=['30','01','02'] #middle date is simulation date
date_folder=MakeDateFolder(date_string)

#running
DownloadERA5(variables,years,months,days,area)

2025-09-03 18:39:28,883 INFO [2024-09-26T00:00:00] Watch our [Forum](https://forum.ecmwf.int/) for Announcements, news and other discussed topics.

2025-09-03 18:39:29,404 INFO Request ID is 1a1223cd-886b-4343-bf4a-ea079691fcf3
2025-09-03 18:39:29,637 INFO status has been updated to accepted
2025-09-03 18:39:39,649 INFO status has been updated to successful


f24e7c866e4c8d6da9bbbec4d22e22e9.nc:   0%|          | 0.00/768k [00:00<?, ?B/s]

2025-09-03 18:39:43,247 INFO Request ID is 90d344c1-f9cd-438f-9c6e-f483d48c7b97
2025-09-03 18:39:43,534 INFO status has been updated to accepted
2025-09-03 18:40:05,743 INFO status has been updated to running
2025-09-03 18:41:39,231 INFO status has been updated to successful


c85aa57ba1107a0d9f6cd90181e89c80.nc:   0%|          | 0.00/770k [00:00<?, ?B/s]

...

a4057f206caf8cf7063313023592d6c8.nc:   0%|          | 0.00/722k [00:00<?, ?B/s]

2025-09-03 18:55:54,972 INFO Request ID is 6db55581-4a7b-4660-94f3-5d5a79f89b3f
2025-09-03 18:55:55,216 INFO status has been updated to accepted
2025-09-03 18:56:05,275 INFO status has been updated to running
2025-09-03 18:57:13,283 INFO status has been updated to successful


86e46b748eeecbf73d7823cd9f89b1cd.nc:   0%|          | 0.00/175k [00:00<?, ?B/s]

2025-09-03 18:57:16,629 INFO Request ID is c386f19b-322c-4f35-8724-b4702bd85a89
2025-09-03 18:57:16,872 INFO status has been updated to accepted
2025-09-03 18:57:31,186 INFO status has been updated to running
2025-09-03 18:59:12,691 INFO status has been updated to successful


6034dfbfd8868d4438458d3f1fa9594c.nc:   0%|          | 0.00/506k [00:00<?, ?B/s]

In [55]:
###########################
#DATE THREE (INTERESTING CASE)

In [56]:
#INFORMATION
#date information
date_string = "08-11 - 08-13 (2022)"
years=['2022']
months=['08']
days=['11','12','13'] #middle date is simulation date
date_folder=MakeDateFolder(date_string)

#running
DownloadERA5(variables,years,months,days,area)

2025-09-03 18:59:16,602 INFO [2024-09-26T00:00:00] Watch our [Forum](https://forum.ecmwf.int/) for Announcements, news and other discussed topics.

2025-09-03 18:59:17,172 INFO Request ID is 114071da-ead1-4f3d-a25e-242f705f31d6
2025-09-03 18:59:17,417 INFO status has been updated to accepted
2025-09-03 18:59:31,829 INFO status has been updated to running
2025-09-03 19:00:08,581 INFO status has been updated to successful


2e5362985de29cd68b641265de93e2ab.nc:   0%|          | 0.00/397k [00:00<?, ?B/s]

2025-09-03 19:00:12,087 INFO Request ID is eb75aa64-1556-4852-bdd5-616ba0759371
2025-09-03 19:00:12,406 INFO status has been updated to accepted
2025-09-03 19:00:46,365 INFO status has been updated to running
2025-09-03 19:01:03,761 INFO status has been updated to successful


ce737c777703f59ae73d0c9966d0eb41.nc:   0%|          | 0.00/392k [00:00<?, ?B/s]

...

6de68652424a430958aaaa0fa493f67.nc:   0%|          | 0.00/367k [00:00<?, ?B/s]

2025-09-03 19:10:16,275 INFO Request ID is d7197880-8622-4b90-83dc-48099193b0f3
2025-09-03 19:10:16,509 INFO status has been updated to accepted
2025-09-03 19:10:25,656 INFO status has been updated to running
2025-09-03 19:11:07,712 INFO status has been updated to successful


94164a7dfa9a742843dd068a3790c3a8.nc:   0%|          | 0.00/116k [00:00<?, ?B/s]

2025-09-03 19:11:10,692 INFO Request ID is 6b5a2dec-8b0d-4b83-a15a-12aa5473a4fb
2025-09-03 19:11:10,987 INFO status has been updated to accepted
2025-09-03 19:11:25,325 INFO status has been updated to running
2025-09-03 19:12:02,082 INFO status has been updated to successful


fe6657b93c9d0dce41ce410861854c44.nc:   0%|          | 0.00/261k [00:00<?, ?B/s]